# Programme 2 - Recalage : coins & normalisation de page
Traitement **bas niveau** (consigne 4.1) : filtrage de Sobel, seuillage d'Otsu, morphologie mathematique, puis detection des 4 reperes en L par **correlation** avec un gabarit (et ses rotations), enfin recalage par homographie (`ProjectiveTransform`) vers une page canonique 1654x2339. Aucune fonction haut niveau de detection de rectangles n'est utilisee.

In [ ]:
# --- Choix du dossier, du PDF et de la page ------------------------------
# dossier    : "FORM1" / "FORM2" / "FORM3"
# page_index : 0 = page d'identification (PAGE-01) ; 4 = 1re page de questions
import os, glob, numpy as np
import reader_pdf as R

dossier    = "FORM1"
page_index = 0

pdfs = sorted(glob.glob(os.path.join(dossier, "*.pdf")))
if not pdfs:
    raise FileNotFoundError(
        f"Aucun PDF dans '{os.path.abspath(dossier)}'. "
        f"Lance Jupyter depuis la racine du projet, ou mets un chemin complet.")
pdf_path = pdfs[0]
print("PDF utilise :", pdf_path, "| total trouves :", len(pdfs))

pages = R.pdf_to_pages(pdf_path, dpi=200)   # 1 image par page (pdf2image -> fitz en secours)
im    = pages[page_index]
print("nb pages :", len(pages), "| page", page_index, "shape", im.shape)


## Pretraitement (identique a vos notebooks images)

In [ ]:
import matplotlib.pyplot as plt
# rgb2gray -> sobel -> seuil Otsu*0.7 -> remove_small_objects -> dilation -> closing -> fill_holes
im2, im4, imlabel, regionprops = R.preprocess(im)
print("nb composantes connexes :", len(regionprops))
plt.figure(figsize=(6,8)); plt.imshow(im4, cmap='gray'); plt.title("Masque binaire (im4)"); plt.axis('off'); plt.show()

## Detection des 4 coins (correlation d'un gabarit en L + rotations)

In [ ]:
corners = R.find_corners(imlabel, regionprops)
print("coins (x,y) TL,TR,BL,BR :", corners)
plt.figure(figsize=(6,8)); plt.imshow(im2, cmap='gray')
if corners:
    xs=[p[0] for p in corners]; ys=[p[1] for p in corners]
    plt.scatter(xs, ys, c='r', s=80, marker='+')
plt.title("Reperes detectes"); plt.axis('off'); plt.show()

## Recalage (homographie) vers la page canonique

In [ ]:
g = R.normalize_page(im, corners)   # page normalisee, niveaux de gris [0,1]
print("page normalisee :", g.shape)
plt.figure(figsize=(6,8)); plt.imshow(g, cmap='gray'); plt.title("Page recalee"); plt.axis('off'); plt.show()

> **Garde-fou** : sur une page blanche (ex. page 2), `find_corners` renvoie `None` ; `normalize_page` bascule alors sur un simple redimensionnement, sans planter.